# This Evaluates the Retrieval and Response Generation of the RAG model

1. Embed each question in the test set
2. Retrieve the top 3 documents using the RAG retrieval function
3. Compare top-1 retrieved filename to the ground truth file column
4. Score as 1 if correct, else 0
5. Compute the mean score

In [7]:
import pandas as pd
import numpy as np
from numpy.linalg import norm
from tqdm import tqdm
from utils import get_embedding, df as embeddings_df

# Load test set
testset = pd.read_excel("testset.xlsx")

# Define safe division function
def safe_divide(a, b):
    return a / b if b != 0 else 0

# Function to compute cosine similarity
def compute_similarity(vec1, vec2):
    return np.dot(vec1, vec2) / safe_divide(norm(vec1) * norm(vec2), 1)

# Store results
results = []

# Evaluate retrieval for each question
for idx, row in tqdm(testset.iterrows(), total=len(testset)):
    question = row["question"]
    expected_file = row["file"]

    # Embed the question
    user_embedding = get_embedding(question, model='text-embedding-3-small')
    user_embedding_np = np.array(user_embedding)

    # Compute cosine similarities
    similarities = embeddings_df['embedding'].apply(
        lambda x: compute_similarity(x, user_embedding_np)
    )

    # Get top 3 most similar documents
    top_indices = similarities.nlargest(3).index
    top_filenames = embeddings_df.loc[top_indices, 'filename'].tolist()

    # Evaluate top-N retrieval
    is_correct_top1 = int(expected_file == top_filenames[0])
    is_correct_top2 = int(expected_file in top_filenames[:2])
    is_correct_top3 = int(expected_file in top_filenames[:3])

    # Save the result
    results.append({
        "question": question,
        "expected_file": expected_file,
        "top1_file": top_filenames[0],
        "top2_file": top_filenames[1] if len(top_filenames) > 1 else "",
        "top3_file": top_filenames[2] if len(top_filenames) > 2 else "",
        "top3_files": top_filenames,
        "is_correct_top1": is_correct_top1,
        "is_correct_top2": is_correct_top2,
        "is_correct_top3": is_correct_top3
    })

# Create a results DataFrame
results_df = pd.DataFrame(results)

# Calculate mean scores
top1_accuracy = results_df["is_correct_top1"].mean()
top2_accuracy = results_df["is_correct_top2"].mean()
top3_accuracy = results_df["is_correct_top3"].mean()

# Print results
print(f"Top-1 Retrieval Accuracy: {top1_accuracy:.2f}")
print(f"Top-2 Retrieval Accuracy: {top2_accuracy:.2f}")
print(f"Top-3 Retrieval Accuracy: {top3_accuracy:.2f}")

# Save results to CSV
results_df.to_csv("retrieval_evaluation_results.csv", index=False)
print("Detailed results saved to retrieval_evaluation_results.csv")

# Display DataFrame inline
try:
    import ace_tools as tools
    tools.display_dataframe_to_user(name="Retrieval Evaluation Results", dataframe=results_df)
except ImportError:
    from IPython.display import display
    display(results_df.head())


100%|██████████| 76/76 [00:24<00:00,  3.08it/s]

Top-1 Retrieval Accuracy: 0.76
Top-2 Retrieval Accuracy: 0.84
Top-3 Retrieval Accuracy: 0.89
Detailed results saved to retrieval_evaluation_results.csv


,question,expected_file,top1_file,top2_file,top3_file,top3_files,is_correct_top1,is_correct_top2,is_correct_top3
0,Um was geht es hier?,01-introduction.qmd,07-structure.qmd,14-poster.qmd,10-writing.qmd,"[07-structure.qmd, 14-poster.qmd, 10-writing.qmd]",0,0,0
1,Wie sieht der Alltag eines Forschers aus?,01-introduction.qmd,01-introduction.qmd,02-researchprocess.qmd,07-structure.qmd,"[01-introduction.qmd, 02-researchprocess.qmd, ...",1,1,1
2,Was ist der Unterschied zwischen Grundlagenfor...,01-introduction.qmd,01-introduction.qmd,10-writing.qmd,02-researchprocess.qmd,"[01-introduction.qmd, 10-writing.qmd, 02-resea...",1,1,1
3,Was ist der Unterschied zwischen Studie und Ex...,02-researchprocess.qmd,02-researchprocess.qmd,04-datacollection.qmd,08-content.qmd,"[02-researchprocess.qmd, 04-datacollection.qmd...",1,1,1
4,Was ist der Unterschied zwischen Reproduzierba...,02-researchprocess.qmd,02-researchprocess.qmd,08-content.qmd,08-content.qmd,"[02-researchprocess.qmd, 08-content.qmd, 08-co...",1,1,1
